# mapless40 — 코랩에서 학습하기

40 Hz mapless asymmetric SAC 정책을 무료 코랩 GPU로 학습하고, Jetson 에 넣을 파일(`actor.ts.pt`, `actor_meta.json`)을 뽑는다.

**처음 한 번:** 위 메뉴 `런타임 → 런타임 유형 변경 → T4 GPU` 선택 후, 위에서부터 셀을 차례로 실행.

**세션이 끊기면:** 다시 열어서 ① ~ ③ 셀과 ⑥ 학습 셀만 다시 실행하면 구글 드라이브에 저장된 체크포인트에서 이어서 학습한다.

## ① GPU 확인

In [ ]:
!nvidia-smi -L || echo "GPU 없음 → 런타임 유형을 T4 GPU 로 바꾸세요"
!python --version

## ② 구글 드라이브 연결 (체크포인트 보관용)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RUN_DIR = '/content/drive/MyDrive/mapless40_runs/conv1d_main'   # 실험마다 이름을 바꾸면 따로 저장됨
import os; os.makedirs(RUN_DIR, exist_ok=True); print(RUN_DIR)

## ③ 코드 받기 + 패키지 설치 (1~2분)

In [ ]:
%cd /content
!test -d 2027_F1tenth_test || git clone -q --depth 1 -b feat/mapless40-asym-sac https://github.com/tkddn647-ship-it/2027_F1tenth_test
%cd /content/2027_F1tenth_test
!git pull -q origin feat/mapless40-asym-sac
!git log --oneline -1
!pip -q install "stable-baselines3>=2.3" "gymnasium>=0.29" scipy pyyaml pillow tensorboard

## ④ 테스트 (먼저 꼭 실행)

numpy 테스트 + torch 테스트(actor 가 privileged 정보를 안 보는지, 짧은 학습 루프가 도는지, 커넥톰 수정 확인).
**FAIL 이 나오면 여기서 멈추고 출력 전체를 복사해서 알려주세요.**

In [ ]:
!python -m mapless40.tests --quick

## ⑤ 기준선 확인 (선택, 약 1분) — FTG 가 ifac 를 몇 초에 도는지

In [ ]:
!python -m mapless40.evaluate --baseline ftg --maps ifac --spawns 2 --laps 1 --out /content/eval_ftg
from IPython.display import Image, display
display(Image('/content/eval_ftg/ftg_ifac.png', width=500))

## ⑥ 학습

| 설정 | 의미 |
|--|--|
| `MAPS` | 학습 트랙. ifac(실제 대회 트랙) + F1 트랙 몇 개 |
| `EVAL_MAPS` | 평가 트랙. ifac + 학습에 안 쓴 Budapest(일반화 확인) |
| `STEPS_PER_SESSION` | 이번 세션에 **추가로** 학습할 스텝 수 |
| `TIME_LIMIT_MIN` | 이 시간이 지나면 저장하고 종료 (무료 코랩은 몇 시간 뒤 끊김) |

- 로그의 `[eval]` 줄이 핵심: `점수`(라인 따라 전진 속도 m/s, 높을수록 좋음), `충돌률`, `최고랩`.
- 체크포인트는 5만 스텝마다 드라이브에 저장. 끊기면 이 셀만 다시 실행하면 이어서 학습 (`--resume auto`).
- 이어 학습할 때는 리플레이 버퍼가 비어서 처음 5천 스텝은 다시 모은다.

In [ ]:
MAPS = 'ifac,Spielberg,Silverstone,Monza'
EVAL_MAPS = 'ifac,Budapest'
ENCODER = 'conv1d'          # 비교 실험은 'bev' + RUN_DIR 이름 변경
STEPS_PER_SESSION = 1_000_000
TIME_LIMIT_MIN = 150         # 무료 코랩이면 150분 정도 권장
N_ENVS = 4

!python -m mapless40.train --maps {MAPS} --eval-maps {EVAL_MAPS} --encoder {ENCODER} \
   --timesteps {STEPS_PER_SESSION} --n-envs {N_ENVS} --subproc --device cuda \
   --buffer-size 150000 --learning-starts 20000 --gradient-steps 2 \
   --eval-freq 50000 --ckpt-freq 50000 --time-limit-min {TIME_LIMIT_MIN} \
   --save-dir "{RUN_DIR}" --resume auto

## ⑦ 학습 곡선 보기 (학습 중에도 다른 탭에서 확인 가능)

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
df = pd.read_csv(f'{RUN_DIR}/eval.csv')
g = df.groupby(['timesteps','map']).agg(score=('score_mps','mean'), crash=('collided','mean'), lap=('best_lap','min')).reset_index()
fig, ax = plt.subplots(1, 3, figsize=(15, 3.5))
for m, d in g.groupby('map'):
    ax[0].plot(d.timesteps, d.score, marker='o', label=m); ax[1].plot(d.timesteps, d.crash, marker='o', label=m); ax[2].plot(d.timesteps, d.lap, marker='o', label=m)
for a, t in zip(ax, ['점수: 라인 따라 전진 속도 [m/s] ↑', '충돌률 ↓', '최고 랩타임 [s] ↓']):
    a.set_title(t); a.set_xlabel('timesteps'); a.grid(alpha=.3); a.legend()
plt.tight_layout(); plt.show()
g.tail(10)

## ⑧ 최고 모델 평가 — 궤적 그림 + 랩타임 (ifac, Budapest)

In [ ]:
!python -m mapless40.evaluate --model "{RUN_DIR}/best_model.zip" --maps ifac,Budapest --spawns 3 --laps 2 --out /content/eval_best
import glob
from IPython.display import Image, display
for f in sorted(glob.glob('/content/eval_best/*.png')): display(Image(f, width=520))

## ⑨ CNN 이 무엇을 배웠는지 보기 (학습 전과 비교)

In [ ]:
!python -m mapless40.viz_encoder --map ifac --model "{RUN_DIR}/best_model.zip" --out /content/viz_trained --gif-steps 120
from IPython.display import Image, display
display(Image('/content/viz_trained/enc1d_ifac.png', width=1000))

## ⑩ Jetson 용 파일 내보내기 + 다운로드

`actor.ts.pt` 와 `actor_meta.json` 두 파일만 Jetson 레포 루트에 복사한 뒤:

```bash
python3 -m mapless40.ros_node --ros-args -p model:=actor.ts.pt -p meta:=actor_meta.json -p max_speed:=2.0
```

첫 실차 시험은 `max_speed:=2.0` 으로. 시뮬 평가(⑧)에서 FTG 기준선보다 빠르고 충돌이 없을 때만 올릴 것.

In [ ]:
!python -m mapless40.export "{RUN_DIR}/best_model.zip" --out-dir "{RUN_DIR}/export" --onnx
!python -m mapless40.evaluate --model "{RUN_DIR}/export/actor.ts.pt" --maps ifac --spawns 1 --laps 1 --out /content/eval_ts
from google.colab import files
files.download(f'{RUN_DIR}/export/actor.ts.pt'); files.download(f'{RUN_DIR}/export/actor_meta.json')